In [1]:
import requests
import pandas as pd
import pymysql

conn = pymysql.connect(
    host="localhost",
    user="root",
    password="root",
    database="nhl",
)

try:
    teams_df = pd.read_sql(
        "SELECT team_id, team_abbrev FROM teams ORDER BY team_abbrev",
        conn,
    ).dropna(subset=["team_abbrev"])
finally:
    conn.close()

game_records = []

for team in teams_df.itertuples(index=False):
    url = f"https://api-web.nhle.com/v1/club-schedule-season/{team.team_abbrev}/20252026"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        schedule = response.json()
    except requests.RequestException as error:
        print(f"Could not load schedule for {team.team_abbrev}: {error}")
        continue

    for game in schedule.get("games", []):
        home_team = game.get("homeTeam", {})
        away_team = game.get("awayTeam", {})
        venue = game.get("venue", {})
        game_records.append({
            "game_id": game.get("id"),
            "season": game.get("season"),
            "game_type": game.get("gameType"),
            "game_date": game.get("gameDate"),
            "home_team_id": home_team.get("id"),
            "away_team_id": away_team.get("id"),
            "home_score": home_team.get("score"),
            "away_score": away_team.get("score"),
            "game_state": game.get("gameState"),
            "venue_name": venue.get("default", ""),
        })

schedule_df = pd.DataFrame(game_records).drop_duplicates("game_id")
schedule_df = schedule_df[
    [
        "game_id",
        "season",
        "game_type",
        "game_date",
        "home_team_id",
        "away_team_id",
        "home_score",
        "away_score",
        "game_state",
        "venue_name",
    ]
].sort_values("game_date").reset_index(drop=True)
schedule_df

C:\Users\nihaa\AppData\Local\Temp\ipykernel_6112\2285094515.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  teams_df = pd.read_sql(


,game_id,season,game_type,game_date,home_team_id,away_team_id,home_score,away_score,game_state,venue_name
0,2025010101,20252026,1,2025-09-20,25,19,2,1,FINAL,American Airlines Center
1,2025010001,20252026,1,2025-09-21,26,24,3,1,FINAL,Toyota Arena
2,2025010003,20252026,1,2025-09-21,20,22,0,3,FINAL,Scotiabank Saddledome
3,2025010002,20252026,1,2025-09-21,22,20,2,3,FINAL,Rogers Place
4,2025010004,20252026,1,2025-09-21,18,13,5,0,FINAL,Bridgestone Arena
...,...,...,...,...,...,...,...,...,...,...
1493,2025030412,20252026,3,2026-06-04,12,54,4,3,OFF,Lenovo Center
1494,2025030413,20252026,3,2026-06-06,54,12,5,4,OFF,T-Mobile Arena
1495,2025030414,20252026,3,2026-06-09,54,12,3,5,OFF,T-Mobile Arena
1496,2025030415,20252026,3,2026-06-11,12,54,4,2,OFF,Lenovo Center


In [2]:
conn = pymysql.connect(
    host="localhost",
    user="root",
    password="root",
    database="nhl",
)

try:
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS games (
            game_id BIGINT PRIMARY KEY,
            season VARCHAR(20) NOT NULL,
            game_type INT,
            game_date DATE,
            home_team_id INT,
            away_team_id INT,
            home_score INT,
            away_score INT,
            game_state VARCHAR(20),
            venue_name VARCHAR(255)
        )
    """)

    game_columns = [
        "game_id",
        "season",
        "game_type",
        "game_date",
        "home_team_id",
        "away_team_id",
        "home_score",
        "away_score",
        "game_state",
        "venue_name",
    ]
    game_records = list(
        schedule_df[game_columns]
        .astype(object)
        .where(pd.notna(schedule_df[game_columns]), None)
        .itertuples(index=False, name=None)
    )

    insert_query = """
        INSERT INTO games (
            game_id, season, game_type, game_date, home_team_id,
            away_team_id, home_score, away_score, game_state, venue_name
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            season = VALUES(season),
            game_type = VALUES(game_type),
            game_date = VALUES(game_date),
            home_team_id = VALUES(home_team_id),
            away_team_id = VALUES(away_team_id),
            home_score = VALUES(home_score),
            away_score = VALUES(away_score),
            game_state = VALUES(game_state),
            venue_name = VALUES(venue_name)
    """

    cursor.executemany(insert_query, game_records)
    conn.commit()
    print(f"Loaded {len(game_records)} games into the games table")
finally:
    conn.close()

Loaded 1498 games into the games table
